In [ ]:
!pip install --upgrade transformers -q


In [ ]:
import json, random, torch, numpy as np
from datetime import datetime
from datasets import Dataset, load_dataset, concatenate_datasets, Value
from transformers import AutoTokenizer, pipeline
from transformers import logging as hf_logging
from huggingface_hub import login, HfApi, create_repo, hf_hub_download, repo_exists
import os
import glob
from tqdm.auto import tqdm
hf_logging.set_verbosity_error()


def get_secret(label, filename="secrets.json"):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        pass

    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches:
        with open(matches[0]) as f:
            return json.load(f)[label]

    listing = os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "MISSING"
    raise ValueError(f"Secret '{label}' not found. /kaggle/input contains: {listing}")

HF_USER =           get_secret("HF_USER")
HF_TOKEN =          get_secret("HF_TOKEN")
SYNTH_DATA_REPO =   HF_USER + "/" + get_secret("SYNTH_DATA_REPO")

login(token=HF_TOKEN)

COMPANY       = get_secret("COMPANY")
COMPANY_DESC  = get_secret("COMPANY_DESC")
BASE_MODEL    = "cardiffnlp/twitter-roberta-base-sentiment-latest"
N_PER_CLASS   = 100
LABEL_MAP     = {"negative": 0, "neutral": 1, "positive": 2}


In [ ]:
MAX_COLLECTED_POSTS = 400
try:
    COLLECTED_DATA = hf_hub_download(repo_id=f"{HF_USER}/sentiment_posts", filename="new_posts.json", repo_type="dataset")
    collected_posts = json.load(open(COLLECTED_DATA))
    collected_text = [post["text"] for post in collected_posts]
    if len(collected_text) > MAX_COLLECTED_POSTS:
        collected_text = random.sample(collected_text, MAX_COLLECTED_POSTS)
except Exception:
    print("No collected posts yet, skipping real-post examples for this run")
    collected_text = []


In [ ]:
generator = pipeline(
    "text-generation",
    model="google/gemma-4-e2b-it",
    dtype=torch.float16,
    device=0,
    max_length=None,
    max_new_tokens=80,
    do_sample=True,
    temperature=1.2,
    top_p= 0.92,
    top_k=50,
    repetition_penalty=1.3
)

def build_prompt(label):
    examples = random.choices(collected_text, k = 5) if collected_text else []

    templates = {
        "positive": (f"Write a short positive customer review or social media post about "
                    f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                    f"here some stylistic examples with various sentiments{examples}"),

        "negative": (f"Write a short negative complaint or post about "
                    f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                    f"here some stylistic examples with various sentiments{examples}"),

        "neutral":  (f"Write a short neutral news update or factual statement about" 
                    f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                    f"here some stylistic examples with various sentiments{examples}"),
    }
    return templates[label]




def generate_data(label, n, batch_size=20):
    data = []
    with tqdm(total=n, desc=label) as pbar:
        for _ in range(0, n, batch_size):
            current_batch = min(batch_size, n - len(data))
            prompts = [build_prompt(label) for _ in range(current_batch)]
            outputs = generator(
                [[{"role": "user", "content": p}] for p in prompts],
            )
            for out in outputs:
                text = out[0]["generated_text"][-1]["content"].strip()
                if text:
                    data.append({"text": text, "label": LABEL_MAP[label]})
            pbar.update(current_batch)
    return data

gen_data = []
for label in LABEL_MAP:
    print(f"Generating {N_PER_CLASS} {label}...")
    gen_data.extend(generate_data(label, N_PER_CLASS))

random.shuffle(gen_data)
print(f"Total: {len(gen_data)}")

print(random.choices(gen_data, k=10))



In [ ]:
def label_real_posts(posts):
    labeled = []
    for text in posts:
        prompt = (
            f"Classify the sentiment of the following post regarding {COMPANY} as exactly one word: "
            f"positive, neutral, or negative. Output only the single word.\n\nPost: {text}"
        )
        response = generator(
            [[{"role": "user", "content": prompt}]],
            do_sample=False, temperature=None, top_p=None, top_k=None,
            max_new_tokens=5,
        )
        raw = response[0][0]["generated_text"][-1]["content"].strip().lower()
        label_text = next((l for l in LABEL_MAP if l in raw), None)
        if label_text:
            labeled.append({"text": text, "label": LABEL_MAP[label_text]})
    return labeled

posts_labeled = label_real_posts(collected_text)


In [ ]:
del generator
torch.cuda.empty_cache()


In [ ]:
for d in gen_data:
    d["source"] = "synthetic"
for d in posts_labeled:
    d["source"] = "weak_labelled"

new_data = Dataset.from_list(gen_data + posts_labeled)


In [ ]:
try:
    existing = load_dataset(SYNTH_DATA_REPO)["train"]
    new_data = concatenate_datasets([existing, new_data])
except Exception:
    create_repo(SYNTH_DATA_REPO, repo_type="dataset", exist_ok=True)

new_data.push_to_hub(SYNTH_DATA_REPO)
